In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm
from sklearn import preprocessing

In [2]:
df = pd.read_csv("data/labelled_data.csv")
df["Date"] = pd.to_datetime(df["Date"])

In [3]:
df.columns 

Index(['Unnamed: 0', 'Date', 'Close', 'High', 'Low', 'Open', 'Volume',
       'lagged_close_1', 'log_returns', 'squared_log_returns',
       '5_day_rolling_vol', '21_day_rolling_vol', 'target_t'],
      dtype='str')

In [4]:
df.drop(columns=["Unnamed: 0"], inplace=True)

In [5]:
train = df[(df["Date"] >= "2010-01-01") & (df["Date"] < "2022-01-01")]
val = df[(df["Date"] >= "2022-01-01") & (df["Date"] < "2024-01-01")]
test = df[(df["Date"] >= "2024-01-01")]

In [6]:
train.head()

,Date,Close,High,Low,Open,Volume,lagged_close_1,log_returns,squared_log_returns,5_day_rolling_vol,21_day_rolling_vol,target_t
0,2010-02-03,82.402016,82.889692,82.161930,82.439526,172730700,82.814659,-0.004995,0.000025,0.011492,0.010240,0.245862
1,2010-02-04,79.858604,81.801798,79.843596,81.764288,356715700,82.402016,-0.031352,0.000983,0.017379,0.012302,0.127947
2,2010-02-05,80.023666,80.188721,78.463106,79.948635,493585800,79.858604,0.002065,0.000004,0.016704,0.012309,0.127242
3,2010-02-08,79.445946,80.526334,79.385923,80.083673,224166900,80.023666,-0.007246,0.000052,0.015553,0.012376,0.160716
4,2010-02-09,80.443802,81.141552,79.731043,80.376275,337820500,79.445946,0.012482,0.000156,0.015624,0.012652,0.138217


In [7]:
# We want to fit and transform the features, which include log_returns, squared_log_returns, 
# 5_day_rolling_vol, and 21_day_rolling_vol
# we need to use a different scaler for target_t, so we can inverse it later for evaluation

feature_scaler = preprocessing.StandardScaler()
target_scaler = preprocessing.StandardScaler()

feature_cols = ["log_returns", "squared_log_returns", "5_day_rolling_vol", "21_day_rolling_vol"]
target_col = "target_t"

trainX = feature_scaler.fit_transform(train[feature_cols])
trainY = target_scaler.fit_transform(train[target_col].values.reshape(-1, 1))

valX = feature_scaler.transform(val[feature_cols])
valY = target_scaler.transform(val[target_col].values.reshape(-1, 1))

testX = feature_scaler.transform(test[feature_cols])
testY = target_scaler.transform(test[target_col].values.reshape(-1, 1))

In [9]:
trainX

array([[-0.51914056, -0.19518399,  0.46320202,  0.2182858 ],
       [-2.9806197 ,  1.88193204,  1.34613226,  0.56631006],
       [ 0.14018896, -0.24003985,  1.24477749,  0.56754072],
       ...,
       [ 0.06674074, -0.24574049, -0.02868906,  0.45217313],
       [-0.3111724 , -0.23266787, -0.20859736,  0.31666512],
       [-0.28830784, -0.23547689, -0.27966596,  0.27215921]],
      shape=(3000, 4))

In [10]:
trainY

array([[ 1.06541388],
       [-0.04950472],
       [-0.05617132],
       ...,
       [ 0.12195374],
       [ 0.11085032],
       [ 0.12609541]], shape=(3000, 1))

In [11]:
lstmXTraining, lstmYTraining = [], []

for i in range(0, len(trainX) - 21):
    lstmXTraining.append(trainX[i:i+21])
    lstmYTraining.append(trainY[i+20])

lstmXTraining = np.array(lstmXTraining)
lstmYTraining = np.array(lstmYTraining)

In [12]:
lstmXTraining

array([[[-0.51914056, -0.19518399,  0.46320202,  0.2182858 ],
        [-2.9806197 ,  1.88193204,  1.34613226,  0.56631006],
        [ 0.14018896, -0.24003985,  1.24477749,  0.56754072],
        ...,
        [ 0.2057513 , -0.23268561, -0.31082518,  0.28336107],
        [ 0.03056556, -0.24756237, -0.53366139,  0.19122599],
        [ 0.22967553, -0.22946978, -0.51139856,  0.13627011]],

       [[-2.9806197 ,  1.88193204,  1.34613226,  0.56631006],
        [ 0.14018896, -0.24003985,  1.24477749,  0.56754072],
        [-0.72930173, -0.13545971,  1.07215885,  0.57885835],
        ...,
        [ 0.03056556, -0.24756237, -0.53366139,  0.19122599],
        [ 0.22967553, -0.22946978, -0.51139856,  0.13627011],
        [ 1.27274814,  0.18741213, -0.04990414,  0.20744111]],

       [[ 0.14018896, -0.24003985,  1.24477749,  0.56754072],
        [-0.72930173, -0.13545971,  1.07215885,  0.57885835],
        [ 1.11304499,  0.08851307,  1.08279986,  0.62535161],
        ...,
        [ 0.22967553, -0.22

In [13]:
lstmYTraining

array([[-0.2164924 ],
       [-0.83514254],
       [-0.83494787],
       ...,
       [-0.78888023],
       [ 0.12195374],
       [ 0.11085032]], shape=(2979, 1))

In [14]:
lstmXVal, lstmYVal = [], []

for i in range(0, len(valX) - 21):
    lstmXVal.append(valX[i:i+21])
    lstmYVal.append(valY[i+20])

lstmXVal = np.array(lstmXVal)
lstmYVal = np.array(lstmYVal)

In [15]:
lstmXVal

array([[[ 0.48651734, -0.17701896, -0.78776356,  0.19524282],
        [-0.08391739, -0.24904032, -0.79041691,  0.1646307 ],
        [-1.86336571,  0.56578914,  0.11971468,  0.25814635],
        ...,
        [ 2.23839834,  1.05555566,  0.63845757,  0.32055207],
        [ 1.61442423,  0.4415878 ,  0.96634576,  0.43237624],
        [ 0.57626481, -0.15095857,  0.8570367 ,  0.44594266]],

       [[-0.08391739, -0.24904032, -0.79041691,  0.1646307 ],
        [-1.86336571,  0.56578914,  0.11971468,  0.25814635],
        [-0.1404254 , -0.24736782,  0.10861981,  0.0896414 ],
        ...,
        [ 1.61442423,  0.4415878 ,  0.96634576,  0.43237624],
        [ 0.57626481, -0.15095857,  0.8570367 ,  0.44594266],
        [ 0.8501708 , -0.04666148,  0.94767371,  0.46668442]],

       [[-1.86336571,  0.56578914,  0.11971468,  0.25814635],
        [-0.1404254 , -0.24736782,  0.10861981,  0.0896414 ],
        [-0.42257373, -0.21526335,  0.12385316,  0.09332522],
        ...,
        [ 0.57626481, -0.15

In [17]:
lstmYVal.shape

(480, 1)

In [18]:
lstmXTesting, lstmYTesting = [], []

for i in range(0, len(testX) - 21):
    lstmXTesting.append(testX[i:i+21])
    lstmYTesting.append(testY[i+20])

lstmXTesting = np.array(lstmXTesting)
lstmYTesting = np.array(lstmYTesting)

In [19]:
lstmXTesting

array([[[-0.57674981, -0.18099716, -0.73632443, -0.46290127],
        [-0.81845957, -0.10348839, -0.55521581, -0.44208781],
        [-0.35394059, -0.22671579, -0.53278097, -0.45312821],
        ...,
        [ 0.68402408, -0.11437712, -0.57994431, -0.50882515],
        [-0.12490424, -0.24798535, -0.60651844, -0.50851619],
        [-1.58906875,  0.33755109,  0.02015021, -0.34435567]],

       [[-0.81845957, -0.10348839, -0.55521581, -0.44208781],
        [-0.35394059, -0.22671579, -0.53278097, -0.45312821],
        [ 0.0751827 , -0.2452217 , -0.52745005, -0.45195152],
        ...,
        [-0.12490424, -0.24798535, -0.60651844, -0.50851619],
        [-1.58906875,  0.33755109,  0.02015021, -0.34435567],
        [ 1.1617335 ,  0.11732062,  0.24563068, -0.26686795]],

       [[-0.35394059, -0.22671579, -0.53278097, -0.45312821],
        [ 0.0751827 , -0.2452217 , -0.52745005, -0.45195152],
        [ 1.27115364,  0.18636203, -0.07578169, -0.33956509],
        ...,
        [-1.58906875,  0.33

In [20]:
lstmYTesting.shape

(475, 1)

# Historical Volatility

In [21]:
hv_predictions = test["5_day_rolling_vol"]
actual_vol = test["target_t"]

## Metrics

### Root Mean Squared Error

In [22]:
from sklearn.metrics import root_mean_squared_error
root_mean_squared_error(actual_vol, hv_predictions)

0.15545281837164687

### Mean Absolute Error

In [23]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(actual_vol, hv_predictions)

0.12303527664366638

### Quasi-Likelihood

In [ ]:
predicted_variance = np.square(hv_predictions)
actual_variance = np.square(actual_vol)
inner_term = predicted_variance / actual_variance

qlike = np.sum(inner_term - np.log(inner_term) - 1) / len(test)
qlike 

np.float64(4.5371696024630985)

# GARCH

In [39]:
from arch.univariate import arch_model

full_series = pd.concat([
    train.set_index("Date")["log_returns"], 
    val.set_index("Date")["log_returns"], 
    test.set_index("Date")["log_returns"]
])

model = arch_model(full_series, mean="Constant", vol="GARCH", p=1, q=1, rescale=True)
model_results = model.fit(last_obs="2021-12-31", disp="off")

# model_results.plot()
forecasts = model_results.forecast(start="2024-01-01", horizon=5)
garch_predictions = np.sqrt(252 * forecasts.variance.mean(axis=1) / 10000)


In [40]:
garch_predictions

Date
2024-01-02    0.109113
2024-01-03    0.119587
2024-01-04    0.115042
2024-01-05    0.108008
2024-01-08    0.135142
                ...   
2025-12-16    0.110721
2025-12-17    0.131341
2025-12-18    0.129252
2025-12-19    0.131402
2025-12-22    0.126584
Length: 496, dtype: float64

## Metrics

### Root Mean Squared Error

In [48]:
actual_vol = test.set_index("Date")['target_t']

In [49]:
root_mean_squared_error(actual_vol, garch_predictions)

0.08936723575515045

### Mean Absolute Error

In [50]:
mean_absolute_error(actual_vol, garch_predictions)

0.05437304830986124

### Quasi-likelihood

In [51]:
predicted_variance = np.square(garch_predictions)
actual_variance = np.square(actual_vol)
inner_term = predicted_variance / actual_variance

qlike = np.sum(inner_term - np.log(inner_term) - 1) / len(test)
qlike 

np.float64(0.8113106226979714)

# LSTM

In [ ]:
import torch

# to use pytorch, we have to convert numpy arrays to tensors
# two ways of doing it, either we do from_numpy() then do .float() as numpy arrays are float64 by default and pytorch expects float32
# second way is to just do FloatTensor(numpy array)

lstmXTraining = torch.from_numpy(lstmXTraining).float()
lstmYTraining = torch.from_numpy(lstmYTraining).float()

lstmXVal = torch.from_numpy(lstmXVal).float()
lstmYVal = torch.from_numpy(lstmYVal).float()

lstmXTesting = torch.FloatTensor(lstmXTesting)
lstmYTesting = torch.FloatTensor(lstmYTesting)

In [53]:
lstmXTraining

tensor([[[-0.5191, -0.1952,  0.4632,  0.2183],
         [-2.9806,  1.8819,  1.3461,  0.5663],
         [ 0.1402, -0.2400,  1.2448,  0.5675],
         ...,
         [ 0.2058, -0.2327, -0.3108,  0.2834],
         [ 0.0306, -0.2476, -0.5337,  0.1912],
         [ 0.2297, -0.2295, -0.5114,  0.1363]],

        [[-2.9806,  1.8819,  1.3461,  0.5663],
         [ 0.1402, -0.2400,  1.2448,  0.5675],
         [-0.7293, -0.1355,  1.0722,  0.5789],
         ...,
         [ 0.0306, -0.2476, -0.5337,  0.1912],
         [ 0.2297, -0.2295, -0.5114,  0.1363],
         [ 1.2727,  0.1874, -0.0499,  0.2074]],

        [[ 0.1402, -0.2400,  1.2448,  0.5675],
         [-0.7293, -0.1355,  1.0722,  0.5789],
         [ 1.1130,  0.0885,  1.0828,  0.6254],
         ...,
         [ 0.2297, -0.2295, -0.5114,  0.1363],
         [ 1.2727,  0.1874, -0.0499,  0.2074],
         [-0.0363, -0.2492, -0.2678, -0.2387]],

        ...,

        [[ 1.0860,  0.0730,  0.4812, -0.3818],
         [-1.8877,  0.5878,  0.9154, -0.1717]

In [ ]:
from torch import nn

class VolatilityLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=4, hidden_size=64, dropout=0.1, num_layers=1)
        self.dropout = nn.Dropout(0.1)
        self.linear = nn.Linear(64, 1)
    
    def forward(self, x):
        output, (hn, cn) = self.lstm(x)
        lastTimestep = output[:, -1, :]
        return self.linear(self.dropout(lastTimestep))



In [57]:
model = VolatilityLSTM()
loss_function = nn.MSELoss()
optimiser = torch.optim.Adam(model.parameters(), lr = 0.001)

max_epochs = 50
patience = 5
delta = 0.001
best_val_loss = float('inf')
no_improvement_count = 0

/Users/sid/Documents/COMP0162 Advanced ML/COMP0162-Advanced-Machine-Learning-Coursework/.venv/lib/python3.13/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [58]:
model.train()
for i in range(max_epochs):
    training_prediction = model(lstmXTraining)
    training_loss = loss_function(training_prediction, lstmYTraining)
    optimiser.zero_grad()
    training_loss.backward()
    optimiser.step()

    model.eval()
    with torch.no_grad():
        val_prediction = model(lstmXVal)
        val_loss = loss_function(val_prediction, lstmYVal)
        print(f"Epoch {i} - Val Loss : {val_loss:.6f}")

        if val_loss < best_val_loss - delta:
            best_val_loss = val_loss
            no_improvement_count = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            no_improvement_count += 1
            if no_improvement_count >= patience:
                print("Stopping early as no improvement has been observed.")
                break
            
    model.train()
model.load_state_dict(torch.load("best_model.pt"))

Epoch 0 - Val Loss : 0.760063
Epoch 1 - Val Loss : 0.741341
Epoch 2 - Val Loss : 0.722982
Epoch 3 - Val Loss : 0.704925
Epoch 4 - Val Loss : 0.686985
Epoch 5 - Val Loss : 0.669075
Epoch 6 - Val Loss : 0.651106
Epoch 7 - Val Loss : 0.632973
Epoch 8 - Val Loss : 0.614506
Epoch 9 - Val Loss : 0.595620
Epoch 10 - Val Loss : 0.576307
Epoch 11 - Val Loss : 0.556497
Epoch 12 - Val Loss : 0.536158
Epoch 13 - Val Loss : 0.515192
Epoch 14 - Val Loss : 0.493583
Epoch 15 - Val Loss : 0.471369
Epoch 16 - Val Loss : 0.448754
Epoch 17 - Val Loss : 0.426109
Epoch 18 - Val Loss : 0.404102
Epoch 19 - Val Loss : 0.383755
Epoch 20 - Val Loss : 0.366551
Epoch 21 - Val Loss : 0.354254
Epoch 22 - Val Loss : 0.348258
Epoch 23 - Val Loss : 0.348496
Epoch 24 - Val Loss : 0.352574
Epoch 25 - Val Loss : 0.357217
Epoch 26 - Val Loss : 0.360418
Epoch 27 - Val Loss : 0.361948
Stopping early as no improvement has been observed.


<All keys matched successfully>

In [59]:
model.eval()

with torch.no_grad():
    testing_predictions = model(lstmXTesting)

testing_predictions

tensor([[ 3.3661e-02],
        [-4.9561e-03],
        [-3.2015e-02],
        [-3.6700e-02],
        [-4.7707e-02],
        [-8.6077e-02],
        [-1.2801e-01],
        [-1.7597e-01],
        [-2.0615e-01],
        [-1.5148e-01],
        [-1.5790e-01],
        [-1.6397e-01],
        [-1.4950e-01],
        [-1.2960e-01],
        [-1.5323e-01],
        [-1.3397e-01],
        [-1.2724e-01],
        [-1.0551e-01],
        [-1.0006e-01],
        [-9.1231e-02],
        [-1.5643e-01],
        [-1.9210e-01],
        [-2.1594e-01],
        [-1.8842e-01],
        [-1.9604e-01],
        [-1.9980e-01],
        [-1.8693e-01],
        [-1.8301e-01],
        [-1.9187e-01],
        [-1.9161e-01],
        [-2.0191e-01],
        [-1.9353e-01],
        [-2.0722e-01],
        [-2.3510e-01],
        [-2.4547e-01],
        [-2.4995e-01],
        [-2.5587e-01],
        [-2.6013e-01],
        [-2.6885e-01],
        [-2.8956e-01],
        [-3.0206e-01],
        [-3.0540e-01],
        [-2.9041e-01],
        [-2

In [64]:
testing_predictions_np = testing_predictions.numpy()
lstmYTesting_np = lstmYTesting.numpy().reshape(-1, 1)

unnormalised_predictions = target_scaler.inverse_transform(testing_predictions_np)
unnormalised_actual = target_scaler.inverse_transform(lstmYTesting_np)

## Metrics

### Root Mean Squared Error

In [65]:
root_mean_squared_error(unnormalised_actual, unnormalised_predictions)

0.08808305114507675

### Mean Absolute Error

In [66]:
mean_absolute_error(unnormalised_actual, unnormalised_predictions)

0.04864004999399185

### Quasi-likelihood

In [67]:
predicted_variance = np.square(unnormalised_predictions)
actual_variance = np.square(unnormalised_actual)
inner_term = predicted_variance / actual_variance

qlike = np.sum(inner_term - np.log(inner_term) - 1) / len(test)
qlike 

np.float32(0.5825195)